**Import libraries**

In [119]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**Load the model-ready dataset**

In [120]:
df = pd.read_csv(
    "../data/processed/model_ready_data.csv"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (101766, 21)


**Check the data**

In [121]:
df.head()

,race,gender,age,admission_type_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,number_inpatient,number_diagnoses,max_glu_serum,A1Cresult,change,diabetesMed,total_previous_visits,total_procedures,readmitted_30_days,patient_nbr
0,Caucasian,Female,[0-10),6,1,1,41,0,1,0,...,0,1,NaN,NaN,No,No,0,41,0,8222157
1,Caucasian,Female,[10-20),1,7,3,59,0,18,0,...,0,9,NaN,NaN,Ch,Yes,0,59,0,55629189
2,AfricanAmerican,Female,[20-30),1,7,2,11,5,13,2,...,1,6,NaN,NaN,No,Yes,3,16,0,86047875
3,Caucasian,Male,[30-40),1,7,2,44,1,16,0,...,0,7,NaN,NaN,Ch,Yes,0,45,0,82442376
4,Caucasian,Male,[40-50),1,7,1,51,0,8,0,...,0,5,NaN,NaN,Ch,Yes,0,51,0,42519267


**Check columns**

In [122]:
print(df.columns.tolist())

# readmitted_30_days = target
# patient_nbr = only for patient grouping
# patient_nbr will NOT be used by the model

['race', 'gender', 'age', 'admission_type_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed', 'total_previous_visits', 'total_procedures', 'readmitted_30_days', 'patient_nbr']


**Separate X and y**

In [123]:
X = df.drop(
    columns=["readmitted_30_days", "patient_nbr"]
)

y = df["readmitted_30_days"]

patient_groups = df["patient_nbr"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (101766, 19)
y shape: (101766,)


**Check target distribution**

In [124]:
print(y.value_counts())

print("\nPercentage:")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# I checked the target distribution because this dataset has a relatively smaller 30-day readmission class, so 
# accuracy alone will not be enough to evaluate the model.

readmitted_30_days
0    90409
1    11357
Name: count, dtype: int64

Percentage:
readmitted_30_days
0    88.84
1    11.16
Name: proportion, dtype: float64


**Patient-level train/test split**

In [125]:
unique_patients = patient_groups.unique()

train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.2,
    random_state=42
)

# We're splitting patients, not individual encounters.
# 80% patients → Training
# 20% patients → Testing

**Create train and test datasets**

In [126]:
train_mask = patient_groups.isin(train_patients)
test_mask = patient_groups.isin(test_patients)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


# Now we have:
# X_train → Training features
# y_train → Training target

# X_test → Testing features
# y_test → Testing target

Training data: (81477, 19)
Testing data: (20289, 19)


**Check the split**

In [127]:
print("Training patients:", len(train_patients))
print("Testing patients:", len(test_patients))

print(
    "\nPatients appearing in both sets:",
    len(
        set(train_patients).intersection(
            set(test_patients)
        )
    )
)

Training patients: 57214
Testing patients: 14304

Patients appearing in both sets: 0


##### I verified that there are zero patients appearing in both the training and testing groups.

**Identify numerical features**

In [128]:
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

Numerical features:
['admission_type_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_previous_visits', 'total_procedures']


**Identify categorical features**

In [129]:
categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

Categorical features:
['race', 'gender', 'age', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']


**Create numerical preprocessing**

In [130]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# For numerical variables, I used median imputation for missing values and standardization so that variables
# with different scales can be used effectively by the model.

**Create categorical preprocessing**

In [131]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# For categorical variables, I filled missing values with the most frequent category and converted categories
# into numerical values using one-hot encoding.

# One-Hot Encoding is a technique used to convert categorical data into numbers so that a machine learning 
# model can understand it.

**Combine preprocessing**

In [132]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

**Create Logistic Regression model**

In [133]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

# I started with Logistic Regression as a baseline model because it is simple and interpretable. I also 
# used balanced class weights to give more importance to the smaller readmission class.

**Train Logistic Regression**

In [134]:
print("Training Logistic Regression...")

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression training completed!")

Training Logistic Regression...
Logistic Regression training completed!


**Make predictions**

In [135]:
logistic_predictions = logistic_model.predict(
    X_test
)

print(
    logistic_predictions[:20]
)

[0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1]


**Evaluate Logistic Regression**

In [136]:
print("Logistic Regression Results")
print("--------------------------------")

print(
    classification_report(
        y_test,
        logistic_predictions
    )
)

Logistic Regression Results
--------------------------------
              precision    recall  f1-score   support

           0       0.92      0.70      0.79     18120
           1       0.16      0.50      0.25      2169

    accuracy                           0.67     20289
   macro avg       0.54      0.60      0.52     20289
weighted avg       0.84      0.67      0.73     20289



**Confusion Matrix**

In [137]:
logistic_cm = confusion_matrix(
    y_test,
    logistic_predictions
)

print(logistic_cm)

[[12602  5518]
 [ 1083  1086]]


## RANDOM FOREST
**Create Random Forest model**

In [138]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1
            )
        )
    ]
)

# I used Random Forest as a second model because it can capture non-linear relationships and interactions 
# between features.

**Train Random Forest**

In [139]:
print("Training Random Forest...")

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed!")

Training Random Forest...
Random Forest training completed!


**Make Random Forest predictions**

In [140]:
random_forest_predictions = (
    random_forest_model.predict(X_test)
)

print(
    random_forest_predictions[:20]
)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


**Evaluate Random Forest**

In [141]:
print("Random Forest Results")
print("--------------------------------")

print(
    classification_report(
        y_test,
        random_forest_predictions
    )
)

Random Forest Results
--------------------------------
              precision    recall  f1-score   support

           0       0.90      0.98      0.94     18120
           1       0.26      0.06      0.10      2169

    accuracy                           0.88     20289
   macro avg       0.58      0.52      0.52     20289
weighted avg       0.83      0.88      0.85     20289



**Compare the two models**

In [142]:
logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions
)

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        logistic_accuracy,
        random_forest_accuracy
    ]
})

comparison

# I'm not automatically choose the model with the highest accuracy.
# Because our target is imbalanced, recall and F1-score for the 30-day readmission class are also very important.
# We'll properly compare these in Notebook 5.

,Model,Accuracy
0,Logistic Regression,0.674651
1,Random Forest,0.880132


**Save the models**

In [143]:
# Create a folder to save the trained models
from pathlib import Path

models_folder = Path("../models")

models_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [144]:
# Save the trained models using joblib
import joblib

joblib.dump(
    logistic_model,
    models_folder / "logistic_regression_model.pkl"
)

joblib.dump(
    random_forest_model,
    models_folder / "random_forest_model.pkl"
)

print("Models saved successfully!")

Models saved successfully!


**Final check**

In [145]:
print("Models trained successfully!")

print("\nLogistic Regression Accuracy:",
      round(logistic_accuracy, 4))

print("Random Forest Accuracy:",
      round(random_forest_accuracy, 4))

Models trained successfully!

Logistic Regression Accuracy: 0.6747
Random Forest Accuracy: 0.8801


## Conclusion

In this notebook, I prepared the data for machine learning and trained two classification models.

Key steps completed:

- Loaded the model-ready dataset.
- Separated the features and target variable.
- Kept patient IDs separately to avoid patient-level data leakage.
- Split the data into training and testing sets based on patients.
- Identified numerical and categorical features.
- Created preprocessing pipelines for missing values, scaling, and categorical encoding.
- Trained a Logistic Regression model as a baseline.
- Trained a Random Forest model as a second machine learning approach.
- Generated predictions for the test dataset.
- Compared the initial model results.
- Saved both trained models for future use.

In the next notebook, I will evaluate the models in more detail using accuracy, precision, recall, F1-score, confusion matrices, ROC-AUC, and other evaluation techniques to determine which model performs better for predicting 30-day hospital readmission.